In [1]:
# %%
import os
import re

print("LinkedIn ETL dependencies loaded.")

LinkedIn ETL dependencies loaded.


In [2]:
# %%
# --- Pipeline Configuration ---
# Path to the file where you manually pasted the LinkedIn posts
INPUT_FILE = "../research/other/raw_linkedin_dump.txt"

# Where the final, formatted markdown files will go
OUTPUT_DIR = "../research/linkedin-posts"

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# %%
def process_linkedin_dump(input_path, output_dir):
    if not os.path.exists(input_path):
        print(f"Error: Could not find raw data file at {input_path}")
        return

    with open(input_path, 'r', encoding='utf-8') as file:
        raw_content = file.read()

    # Split the document into individual posts using our chosen delimiter
    raw_posts = raw_content.split("===END===")
    
    processed_count = 0

    for idx, raw_post in enumerate(raw_posts):
        post_text = raw_post.strip()
        if not post_text:
            continue # Skip any empty blocks (like at the very end of the file)

        try:
            # 1. EXTRACT METADATA using regex
            # Looks for "AUTHOR: [name]" and captures the name
            author_match = re.search(r"AUTHOR:\s*(.+)", post_text)
            url_match = re.search(r"URL:\s*(.+)", post_text)
            
            # The actual post is everything after "POST:"
            content_split = post_text.split("POST:")
            
            if not author_match or len(content_split) < 2:
                print(f"⚠️ Warning: Could not parse block {idx + 1}. Check your formatting.")
                continue
                
            author = author_match.group(1).strip()
            url = url_match.group(1).strip() if url_match else "No URL provided"
            content = content_split[1].strip()

            # 2. TRANSFORM
            # Clean the author name for the filename (e.g., "Josh Braun" -> "josh_braun")
            safe_author = re.sub(r'[^\w\s-]', '', author).strip().replace(' ', '_').lower()
            
            # 3. LOAD
            filename = os.path.join(output_dir, f"{safe_author}_post_{idx + 1}.md")
            
            with open(filename, "w", encoding="utf-8") as f:
                f.write(f"# LinkedIn Post: {author}\n\n")
                f.write(f"**Source URL:** {url}\n\n")
                f.write("---\n\n")
                f.write(content)
                
            print(f"✓ Saved: {filename}")
            processed_count += 1

        except Exception as e:
            print(f"✗ Failed to process block {idx + 1}: {e}")

    print(f"\nETL Complete: Successfully processed {processed_count} posts.")

# Execute the pipeline
process_linkedin_dump(INPUT_FILE, OUTPUT_DIR)

✓ Saved: ../research/linkedin-posts\josh_braun_post_1.md
✓ Saved: ../research/linkedin-posts\guillaume_moubeche_post_2.md
✓ Saved: ../research/linkedin-posts\jason_bay_post_3.md
✓ Saved: ../research/linkedin-posts\alex_berman_post_4.md
✓ Saved: ../research/linkedin-posts\jeremy_chatelaine_post_5.md
✓ Saved: ../research/linkedin-posts\john_barrows_post_6.md
✓ Saved: ../research/linkedin-posts\nick_abraham_post_7.md
✓ Saved: ../research/linkedin-posts\becc_holland_post_8.md
✓ Saved: ../research/linkedin-posts\will_allred_post_9.md
✓ Saved: ../research/linkedin-posts\jed_mahrle_post_10.md

ETL Complete: Successfully processed 10 posts.
